# ReSnap v5a: tambah data rimpang, training ulang

**Satu perubahan saja dibanding v4b:** data train ditambah foto jahe, kunyit, dan lengkuas dari dataset Roboflow.
Resep training, val, dan holdout Open Images sama persis dengan v4b, jadi selisih skor berasal dari data.

Dasar keputusan (langkah 1 dan 2):
- Kunyit, lengkuas, jahe hanya punya 35 foto train masing-masing; AP50 val kunyit 0.23, lengkuas 0.43.
- Kotak tumpukan di train hanya 171 kotak dan contohnya campur aduk (sebagian tumpukan asli, sebagian label objek yang tidak lengkap), jadi **tidak diubah otomatis** di v5a.

Pembagian data rimpang:
| Sumber | Dipakai untuk |
|---|---|
| soda (train), rimpang10 (train), ayush (tanpa `Wild_turmeric`) | train v5a |
| soda dan rimpang10 (valid + test) | **holdout rimpang** (sumber sama, tidak ikut training) |
| rival (test) | **test eksternal rimpang** (sumber berbeda). Disimpan, **tidak dievaluasi** di notebook ini |

Penjagaan kualitas: foto kembar (dHash, termasuk versi cermin) dengan semua data ReSnap dan antar-sumber dibuang; holdout dan test eksternal dibentuk lebih dulu, lalu foto train yang mirip dengannya dibuang; foto yang menurut v4b berisi bahan ReSnap lain tanpa label dibuang; foto `Wild_turmeric` dibuang (ambigu); jumlah tambahan dibatasi 700 kotak per kelas supaya kelas lain tidak tenggelam.

Syarat v5a diterima (ditetapkan sebelum training, dibanding v4b):
1. mAP50-95 val tidak turun lebih dari 0.5 poin.
2. AP50 holdout rimpang naik minimal 10 poin untuk kunyit **dan** lengkuas.
3. Tidak ada kelas val (≥ 20 objek) yang AP50-nya turun lebih dari 10 poin.
4. AP50 holdout Open Images (kentang, udang, jamur) tidak turun lebih dari 5 poin.
5. Kurva tidak overfit/underfit dan gap train-val tidak lebih dari gap v4b + 0.02.

Kaggle: GPU T4, Internet on. Input: dataset `resnap-v4` **dan** output notebook `ReSnap v5 - Audit Tumpukan Rimpang` (berisi `rimpang_raw/*.zip`).
Perkiraan waktu: 4 sampai 5 jam (kebanyakan training). Jalankan dengan Save & Run All (Commit).

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
import json, math, random, shutil, subprocess, sys, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageOps

assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))
DEVICE = '0'
random.seed(42)

INPUT, OUTPUT, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/resnap')
OUT = OUTPUT / 'hasil_v5a'
for p in (OUT, TMP):
    p.mkdir(parents=True, exist_ok=True)

REPO = TMP / 'resnap-cv'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AffanAbya30/resnap-cv.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from diagnose_fit import analyze_curves, load_results, plot as plot_curves, verdict
from evaluate_export import evaluate

V4B = REPO / 'models/best.pt'
assert dv.sha256_file(V4B) == '1410f11d9c2cdfad074446d1742ed5ec24d0269122bc51ec5ddde69bafc66809', 'bukan v4b'

def first_dir(pred):
    hits = sorted((p for p in INPUT.rglob('*') if p.is_dir() and pred(p)), key=lambda p: len(p.parts))
    return hits[0] if hits else None

DATA = first_dir(lambda p: (p / 'holdout_oi/images').is_dir() and (p / 'train/images').is_dir())
BASE = first_dir(lambda p: (p / 'test/images').is_dir() and (p / 'train/images').is_dir())
assert DATA and BASE, 'Input resnap-v4 belum ditambahkan'
print('Data v4:', DATA, '\nData v3:', BASE)

# Data rimpang mentah dari output notebook audit (zip atau sudah diekstrak Kaggle)
RIMPANG = {}
for key in ('soda', 'rimpang10', 'ayush', 'rival'):
    d = first_dir(lambda p, k=key: p.name == k and (p / 'data.yaml').is_file()) or \
        first_dir(lambda p, k=key: (p / 'data.yaml').is_file() and k in str(p))
    if d is None:
        z = next((p for p in INPUT.rglob(f'{key}.zip')), None)
        assert z, f'{key}.zip tidak ditemukan. Tambahkan Input: output notebook "ReSnap v5 - Audit Tumpukan Rimpang".'
        d = TMP / 'rimpang' / key
        with zipfile.ZipFile(z) as f:
            f.extractall(d)
    RIMPANG[key] = d
print({k: str(v) for k, v in RIMPANG.items()})

from ultralytics import YOLO
v4b = YOLO(str(V4B))
NAMES = dict(enumerate(dv.CLASS_NAMES))
IDX = {v: k for k, v in NAMES.items()}
RIMPANG_IDS = {IDX['ginger'], IDX['turmeric'], IDX['galangal']}
MAP = {'jahe': 'ginger', 'ginger': 'ginger', 'kunyit': 'turmeric', 'turmeric': 'turmeric',
       'lengkuas': 'galangal', 'galangal': 'galangal', 'wild_turmeric': 'AMBIGU'}

## 1. Kumpulkan kandidat rimpang dan ubah label ke kelas ReSnap

In [ ]:
def read_labels(lbl):
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            v = line.split()
            if len(v) < 5:
                continue
            c, nums = int(v[0]), list(map(float, v[1:]))
            if len(nums) == 4:
                cx, cy, w, h = nums
            else:
                xs, ys = nums[0::2], nums[1::2]
                cx, cy, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
            out.append((c, cx, cy, w, h))
    return out

cands = []   # satu dict per foto
for key, root in RIMPANG.items():
    cfg = yaml.safe_load((root / 'data.yaml').read_text())
    names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
    cmap = {i: MAP.get(n.strip().lower().replace(' ', '_').replace('-', '_'), 'negatif') for i, n in enumerate(names)}
    for split in ('train', 'valid', 'test'):
        d = root / split / 'images'
        if not d.is_dir():
            continue
        for p in sorted(d.iterdir()):
            if p.suffix.lower() not in dv.IMG_EXT:
                continue
            raw = read_labels(root / split / 'labels' / f'{p.stem}.txt')
            kinds = {cmap.get(c, 'negatif') for c, *_ in raw}
            lines = [f"{IDX[cmap[c]]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in raw
                     if cmap.get(c) in ('ginger', 'turmeric', 'galangal') and w * h >= 0.0004]
            cands.append(dict(src=key, split=split, img=p, lines=lines, ambigu='AMBIGU' in kinds,
                              negatif_saja=not lines and 'negatif' in kinds))
cdf = pd.DataFrame(cands)
print(cdf.groupby(['src', 'split']).agg(foto=('img', 'size'), berlabel=('lines', lambda s: sum(bool(x) for x in s)),
                                         ambigu=('ambigu', 'sum'), negatif_saja=('negatif_saja', 'sum')))

## 2. Bentuk holdout dan test eksternal dulu, lalu saring train

In [ ]:
log = Counter()
ref = {}
for d in [BASE / 'train', BASE / 'val', BASE / 'test', DATA / 'train', DATA / 'val', DATA / 'holdout_oi']:
    ref.update(dv.hash_folder(d / 'images'))
print('Hash data ReSnap (semua split, hanya untuk cek kembar):', len(ref))

for c in cands:
    c['h'] = dv.dhash(c['img']); c['hm'] = dv.dhash(c['img'], mirror=True)
cands = [c for c in cands if c['h'] is not None]

def is_dup(c, pool, d=6):
    if not pool:
        return False
    return bool(dv.near_duplicates({'a': c['h']}, pool, d)) or bool(dv.near_duplicates({'a': c['hm']}, pool, d))

def take(group, name, require_label=True):
    kept, pool = [], {}
    for c in group:
        if c['ambigu'] or (require_label and not c['lines'] and not c['negatif_saja']):
            log[f'{name}: ambigu/kosong'] += 1; continue
        if is_dup(c, ref):
            log[f'{name}: kembar dengan data ReSnap'] += 1; continue
        if is_dup(c, pool, 4):
            log[f'{name}: salinan dalam kelompok'] += 1; continue
        pool[str(c['img'])] = c['h']; kept.append(c)
    return kept, pool

hold, hold_pool = take([c for c in cands if c['src'] in ('soda', 'rimpang10') and c['split'] in ('valid', 'test')], 'holdout')
ext, ext_pool = take([c for c in cands if c['src'] == 'rival' and c['split'] == 'test'], 'eksternal')
# holdout dan eksternal tidak boleh saling kembar
ext = [c for c in ext if not is_dup(c, hold_pool)]
eval_pool = {**hold_pool, **{str(c['img']): c['h'] for c in ext}}

train_pool, tp = [], {}
group = [c for c in cands if (c['src'] in ('soda', 'rimpang10') and c['split'] == 'train') or c['src'] == 'ayush']
random.shuffle(group)
for c in group:
    if c['ambigu']:
        log['train: Wild_turmeric (ambigu)'] += 1; continue
    if not c['lines'] and not c['negatif_saja']:
        log['train: tanpa kotak'] += 1; continue
    if is_dup(c, ref):
        log['train: kembar dengan data ReSnap'] += 1; continue
    if is_dup(c, eval_pool):
        log['train: kembar dengan holdout/eksternal'] += 1; continue
    if is_dup(c, tp, 4):
        log['train: salinan augmentasi'] += 1; continue
    tp[str(c['img'])] = c['h']; train_pool.append(c)
print(dict(log))
print('holdout:', len(hold), '| eksternal:', len(ext), '| kandidat train:', len(train_pool))

## 3. Buang foto berisi bahan ReSnap lain yang tidak berlabel (dicek dengan v4b)

In [ ]:
def unlabeled(c):
    return dv.unlabeled_objects(v4b, str(c['img']), c['lines'], conf=0.6, min_cover=0.5, ignore=RIMPANG_IDS)

for name, group in (('holdout', hold), ('eksternal', ext), ('train', train_pool)):
    keep = []
    for c in group:
        bad = unlabeled(c)
        if bad:
            log[f'{name}: bahan lain tak berlabel'] += 1
        else:
            keep.append(c)
    group[:] = keep
print(dict(log))

## 4. Pilih tambahan train (maks. 700 kotak per kelas, 200 foto negatif) dan susun dataset v5a

In [ ]:
CAP_BOX, CAP_NEG = 700, 200
added, box_count, neg = [], Counter(), 0
by_src = defaultdict(list)
for c in train_pool:
    by_src[c['src']].append(c)
order = []
while any(by_src.values()):            # bergiliran antar sumber supaya beragam
    for s in list(by_src):
        if by_src[s]:
            order.append(by_src[s].pop())
for c in order:
    cls = Counter(int(l.split()[0]) for l in c['lines'])
    if not cls:
        if neg < CAP_NEG:
            added.append(c); neg += 1
        continue
    if any(box_count[k] < CAP_BOX for k in cls):
        added.append(c); box_count.update(cls)
print('Tambahan train:', len(added), 'foto | kotak:', {NAMES[k]: v for k, v in box_count.items()}, '| negatif:', neg)
print('Per sumber:', Counter(c['src'] for c in added))

D5 = TMP / 'data_v5a'
if D5.exists():
    shutil.rmtree(D5)
def copy_split(src_dir, dst_dir, skip_dup=True):
    (dst_dir / 'images').mkdir(parents=True); (dst_dir / 'labels').mkdir()
    for p in (src_dir / 'images').iterdir():
        if skip_dup and '_dup' in p.stem:
            continue
        shutil.copy2(p, dst_dir / 'images' / p.name)
        lbl = src_dir / 'labels' / f'{p.stem}.txt'
        if lbl.exists():
            shutil.copy2(lbl, dst_dir / 'labels' / lbl.name)
for s in ('train', 'val', 'holdout_oi'):
    copy_split(DATA / s, D5 / s)
def write_items(items, split_dir, prefix):
    (split_dir / 'images').mkdir(parents=True, exist_ok=True); (split_dir / 'labels').mkdir(exist_ok=True)
    for i, c in enumerate(items):
        stem = f"{prefix}_{c['src']}_{i:05d}"
        dv.write_pair(c['img'], c['lines'], split_dir / 'images', split_dir / 'labels', stem)
write_items(added, D5 / 'train', 'rp')
write_items(hold, D5 / 'holdout_rimpang', 'hr')
write_items(ext, D5 / 'test_eksternal_rimpang', 'er')

stats = {s: dv.count_split(D5 / s) for s in ('train', 'val', 'holdout_oi', 'holdout_rimpang', 'test_eksternal_rimpang')}
rows = {s: {k: stats[s]['per_class'][k]['instances'] for k in ('ginger', 'turmeric', 'galangal')} | {'foto': stats[s]['images']}
        for s in stats}
display(pd.DataFrame(rows).T)
dv.save_json(OUT / 'jumlah_data_v5a.json', {'log_penyaringan': dict(log), 'jumlah': stats})

# lembar QA visual: 16 foto tambahan train
tiles = []
for c in random.sample(added, min(16, len(added))):
    im = ImageOps.exif_transpose(Image.open(c['img'])).convert('RGB'); w, h = im.size
    d = ImageDraw.Draw(im)
    for l in c['lines']:
        k, cx, cy, bw, bh = l.split(); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        d.rectangle([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h],
                    outline={21: (240, 180, 0), 22: (230, 90, 0), 23: (200, 0, 120)}[int(k)], width=max(2, w // 150))
    im.thumbnail((280, 260)); t = Image.new('RGB', (280, 280), 'white'); t.paste(im, (0, 20))
    ImageDraw.Draw(t).text((4, 4), f"{c['src']} | {len(c['lines'])} kotak", fill='black'); tiles.append(t)
sheet = Image.new('RGB', (4 * 280, math.ceil(len(tiles) / 4) * 280), 'white')
for i, t in enumerate(tiles):
    sheet.paste(t, ((i % 4) * 280, (i // 4) * 280))
sheet.save(OUT / 'qa_tambahan_train.jpg', quality=82)

Y_TRAIN = dv.write_yaml(D5 / 'data.yaml', D5, 'train/images', 'val/images')
Y_HOI = dv.write_yaml(D5 / 'holdout_oi.yaml', D5, 'holdout_oi/images', 'holdout_oi/images')
Y_HR = dv.write_yaml(D5 / 'holdout_rimpang.yaml', D5, 'holdout_rimpang/images', 'holdout_rimpang/images')

## 5. Training v5a (resep sama persis dengan v4b)

In [ ]:
RUN_PROJECT, RUN_NAME = OUTPUT / 'training', 'resnap_yolov8s_v5a_rimpang'
RUN_DIR = RUN_PROJECT / RUN_NAME
MAX_EPOCHS = 70
RECIPE = dict(
    epochs=MAX_EPOCHS, patience=15, batch=16, imgsz=640, save=True, save_period=-1, cache=False,
    workers=2, pretrained=True, optimizer='AdamW', lr0=0.0003, lrf=0.01, momentum=0.937,
    weight_decay=0.001, warmup_epochs=3.0, seed=42, deterministic=True, cos_lr=False,
    close_mosaic=10, amp=True, plots=True, val=True,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, degrees=10.0, translate=0.1, scale=0.5, shear=0.0,
    perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.0,
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
dv.save_json(RUN_DIR / 'experiment.json', {'base_model': 'yolov8s.pt', 'settings': RECIPE,
                                           'perubahan_dari_v4b': 'tambah data rimpang', 'train_images': stats['train']['images']})
YOLO('yolov8s.pt').train(data=str(Y_TRAIN), project=str(RUN_PROJECT), name=RUN_NAME, exist_ok=True, device=DEVICE, **RECIPE)
V5A = RUN_DIR / 'weights/best.pt'
print('Selesai:', V5A, dv.sha256_file(V5A))

## 6. Bandingkan v4b dan v5a (val, holdout Open Images, holdout rimpang)

In [ ]:
rng = random.Random(42)
sub = TMP / 'train_subset'
if sub.exists():
    shutil.rmtree(sub)
(sub / 'images').mkdir(parents=True); (sub / 'labels').mkdir()
old = sorted(p for p in (DATA / 'train/images').iterdir() if '_dup' not in p.stem)
for p in rng.sample(old, 600):        # subset train lama yang sama untuk kedua model
    shutil.copy2(p, sub / 'images' / p.name)
    shutil.copy2(DATA / 'train/labels' / f'{p.stem}.txt', sub / 'labels' / f'{p.stem}.txt')
Y_SUB = dv.write_yaml(sub / 'data.yaml', sub, 'images', 'images')

MODELS = {'v4b': V4B, 'v5a': V5A}
E = {}
for n, w in MODELS.items():
    E[f'{n}_val'] = evaluate(str(w), Y_TRAIN, 'val', OUT, f'{n}_val', DEVICE)
    E[f'{n}_train'] = evaluate(str(w), Y_SUB, 'val', OUT, f'{n}_train_subset', DEVICE)
    E[f'{n}_hoi'] = evaluate(str(w), Y_HOI, 'val', OUT, f'{n}_holdout_oi', DEVICE)
    E[f'{n}_hr'] = evaluate(str(w), Y_HR, 'val', OUT, f'{n}_holdout_rimpang', DEVICE)
gap = {n: E[f'{n}_train']['mAP50_95'] - E[f'{n}_val']['mAP50_95'] for n in MODELS}

def pc(name):
    return pd.read_csv(OUT / name / 'per_class.csv').set_index('class_name')

overall = pd.DataFrame({n: E[f'{n}_val'] for n in MODELS}).loc[['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']]
overall.loc['gap_train_val'] = [gap[n] for n in MODELS]
overall['delta_poin'] = 100 * (overall['v5a'] - overall['v4b'])
display(overall.style.format(precision=3).set_caption('Val 1.069 foto'))

pv = {n: pc(f'{n}_val') for n in MODELS}
cmp = pd.DataFrame({'objek': pv['v5a']['instances'], 'AP50_v4b': pv['v4b']['AP50'], 'AP50_v5a': pv['v5a']['AP50']})
cmp['delta_poin'] = 100 * (cmp.AP50_v5a - cmp.AP50_v4b)
display(cmp.sort_values('delta_poin').style.format(precision=3).set_caption('AP50 per kelas, val'))

def hold_table(tag, classes):
    t = pd.DataFrame({n: pc(f'{n}_{tag}').reindex(classes)['AP50'].fillna(0) for n in MODELS})
    t['delta_poin'] = 100 * (t['v5a'] - t['v4b'])
    return t
hr = hold_table('holdout_rimpang', ['ginger', 'turmeric', 'galangal'])
hoi = hold_table('holdout_oi', ['potato', 'shrimp', 'mushroom'])
display(hr.style.format(precision=3).set_caption('Holdout rimpang (AP50)'))
display(hoi.style.format(precision=3).set_caption('Holdout Open Images (AP50)'))

rows_ = load_results(RUN_DIR / 'results.csv')
curves = analyze_curves(rows_, MAX_EPOCHS)
fit = verdict(curves, E['v5a_train']['mAP50_95'], E['v5a_val']['mAP50_95'], ref_gap=gap['v4b'] - 0.03)
plot_curves(rows_, OUT / 'kurva_v5a.png', curves['best_epoch'], 'Kurva training v5a (tambah data rimpang)')
print(json.dumps({'kurva': curves, **fit}, indent=1, ensure_ascii=False))
for name, df in (('banding_val', overall), ('banding_per_kelas_val', cmp), ('banding_holdout_rimpang', hr), ('banding_holdout_oi', hoi)):
    df.to_csv(OUT / f'{name}.csv')

## 7. Keputusan (syarat ditulis di atas, sebelum training)

In [ ]:
checks = {
    '1_val_map50_95_tidak_turun': overall.loc['mAP50_95', 'delta_poin'] >= -0.5,
    '2_rimpang_naik_kunyit_lengkuas': hr.loc['turmeric', 'delta_poin'] >= 10 and hr.loc['galangal', 'delta_poin'] >= 10,
    '3_tidak_ada_kelas_val_anjlok': not ((cmp.objek >= 20) & (cmp.delta_poin < -10)).any(),
    '4_holdout_oi_tidak_turun': bool((hoi.delta_poin >= -5).all()),
    '5_fit_seimbang_gap_terjaga': fit['vonis'] == 'seimbang',
}
for k, v in checks.items():
    print(('LULUS ' if v else 'GAGAL ') + k)
ACCEPT = all(checks.values())
anjlok = cmp[(cmp.objek >= 20) & (cmp.delta_poin < -10)].index.tolist()
print('\nKelas val yang anjlok:', anjlok or '-')
print('v5a', 'DITERIMA' if ACCEPT else 'BELUM DITERIMA', f"| gap v4b {gap['v4b']:.3f}, v5a {gap['v5a']:.3f}")
dv.save_json(OUT / 'keputusan_v5a.json', {'diterima': ACCEPT, 'syarat': {k: bool(v) for k, v in checks.items()},
                                         'gap': {k: round(v, 4) for k, v in gap.items()},
                                         'kelas_anjlok': anjlok, 'diagnosa': fit, 'evaluasi': E})

## 8. Simpan hasil

- `hasil_v5a.zip`: laporan kecil, kirim ke chat.
- `best_v5a.pt`: bobot model.
- `dataset_v5a.zip`: dataset lengkap (termasuk holdout rimpang dan test eksternal rimpang) untuk run berikutnya.

In [ ]:
shutil.copy2(V5A, OUTPUT / 'best_v5a.pt')
for f in ('results.csv', 'results.png', 'confusion_matrix_normalized.png', 'labels.jpg'):
    if (RUN_DIR / f).exists():
        shutil.copy2(RUN_DIR / f, OUT / f'train_{f}')
pack = OUTPUT / 'hasil_v5a.zip'
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in ('.json', '.csv', '.png', '.jpg'):
            z.write(p, p.relative_to(OUT))
print(pack, round(pack.stat().st_size / 1e6, 1), 'MB')
shutil.make_archive(str(OUTPUT / 'dataset_v5a'), 'zip', D5)
print('dataset_v5a.zip', round((OUTPUT / 'dataset_v5a.zip').stat().st_size / 1e6), 'MB')
shutil.rmtree(RUN_PROJECT / RUN_NAME / 'weights', ignore_errors=True)   # best sudah disalin, hemat Output
for n in MODELS:
    for tag in ('val', 'train_subset', 'holdout_oi', 'holdout_rimpang'):
        d = OUT / f'{n}_{tag}'
        for p in d.glob('*.jpg'):
            p.unlink()